# Multi-Modal RAG with ColPali — Demo Notebook

This notebook walks through the full pipeline step by step:
1. Download a WHO PDF
2. Render pages to images
3. Embed with ColPali
4. Build FAISS index
5. Retrieve relevant pages for sample queries
6. Generate answers with GPT-4o Vision
7. Visualize ColPali similarity maps

GPU recommended (6–8 GB VRAM). Works on CPU too, just slower.

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────
import sys, os
sys.path.insert(0, '..')   # project root

from dotenv import load_dotenv
load_dotenv('../.env')

import warnings
warnings.filterwarnings('ignore')

print('Ready')

In [ ]:
# ── 1. Download one WHO PDF ────────────────────────────────
import requests
from pathlib import Path

data_dir = Path('../data')
data_dir.mkdir(exist_ok=True)

url  = 'https://iris.who.int/bitstream/handle/10665/369364/9789240074323-eng.pdf'
dest = data_dir / 'WHO_World_Health_Statistics_2023.pdf'

if not dest.exists():
    print('Downloading WHO report ...')
    r = requests.get(url, timeout=120)
    dest.write_bytes(r.content)
    print(f'Saved {dest.stat().st_size / 1e6:.1f} MB')
else:
    print(f'Already downloaded: {dest.name}')

In [ ]:
# ── 2. Render first 30 pages ───────────────────────────────
from src.ingestion.pdf_processor import PDFProcessor

processor = PDFProcessor(dpi=150, max_pages=30)
pages = processor.process_pdf(str(dest))
print(f'Rendered {len(pages)} pages')

# preview a few pages
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 6))
sample_indices = [0, 5, 12, 20]
for ax, idx in zip(axes, sample_indices):
    if idx < len(pages):
        ax.imshow(pages[idx].image)
        ax.set_title(f'Page {idx + 1}', fontsize=9)
        ax.axis('off')
plt.suptitle('Sample rendered pages', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Embed pages with ColPali ────────────────────────────
from src.ingestion.embedder import ColPaliEmbedder

embedder = ColPaliEmbedder(model_name='vidore/colpali-v1.3', batch_size=4)

images = [p.image for p in pages]
embeddings = embedder.embed_images(images)

print(f'Embedded {len(embeddings)} pages')
print(f'Shape of first embedding: {embeddings[0].shape}')  # e.g. (1030, 128)

In [ ]:
# ── 4. Build FAISS index ───────────────────────────────────
from pathlib import Path
from src.ingestion.indexer import DocumentIndex, PageMeta

img_dir = Path('../index/page_images')
img_dir.mkdir(parents=True, exist_ok=True)

index = DocumentIndex(embed_dim=128)
metas = []

for i, (page, emb) in enumerate(zip(pages, embeddings)):
    img_path = img_dir / f'{page.doc_name}_p{page.page_number:04d}.png'
    page.image.save(str(img_path), format='PNG')

    metas.append(PageMeta(
        page_id=i,
        doc_name=page.doc_name,
        doc_path=str(dest.resolve()),
        page_number=page.page_number,
        image_path=str(img_path),
    ))

index.add_pages(metas, embeddings)
index.save('../index/')
print('Index saved')

In [ ]:
# ── 5. Retrieve pages for sample queries ──────────────────
from src.retrieval.retriever import ColPaliRetriever
from PIL import Image

retriever = ColPaliRetriever(index=index, embedder=embedder, top_k=3)

test_queries = [
    'What is the global life expectancy at birth?',
    'Show the chart for maternal mortality trends',
    'Which countries have the lowest vaccination coverage?',
]

for query in test_queries:
    results = retriever.retrieve(query)
    print(f'\nQuery: {query}')
    for r in results:
        print(f'  → {r.citation}  (score: {r.score:.3f})')

In [ ]:
# ── 6. Generate an answer with GPT-4o Vision ──────────────
from src.generation.generator import AnswerGenerator

generator = AnswerGenerator(model='gpt-4o')

query    = 'What does the report say about global under-5 mortality?'
results  = retriever.retrieve(query)

images_for_gen = [Image.open(r.image_path).convert('RGB') for r in results]
citations      = [r.citation for r in results]

answer = generator.generate(
    question=query,
    retrieved_images=images_for_gen,
    page_citations=citations,
)

print('Question:', query)
print('\nAnswer:')
print(answer)

In [ ]:
# ── 7. Visualize ColPali similarity maps ──────────────────
# Shows which image patches are most relevant to each query token
# Requires: pip install colpali-engine[interpretability]

try:
    from colpali_engine.interpretability import (
        get_similarity_maps_from_embeddings,
        plot_all_similarity_maps,
    )
    import torch

    query_vis = 'maternal mortality'
    top_result = retriever.retrieve(query_vis)[0]
    page_img   = Image.open(top_result.image_path).convert('RGB')

    batch_imgs = embedder.processor.process_images([page_img]).to(embedder.model.device)
    batch_q    = embedder.processor.process_queries([query_vis]).to(embedder.model.device)

    with torch.no_grad():
        img_emb = embedder.model(**batch_imgs)
        q_emb   = embedder.model(**batch_q)

    n_patches = embedder.processor.get_n_patches(
        image_size=page_img.size, patch_size=embedder.model.patch_size
    )
    img_mask = embedder.processor.get_image_mask(batch_imgs)
    sim_maps = get_similarity_maps_from_embeddings(img_emb, q_emb, n_patches, img_mask)

    tokens = embedder.processor.tokenizer.tokenize(query_vis)
    plots  = plot_all_similarity_maps(page_img, tokens, sim_maps[0])

    fig, axes = plt.subplots(1, len(plots), figsize=(5 * len(plots), 5))
    for ax, (f, _), tok in zip(axes if len(plots) > 1 else [axes], plots, tokens):
        f.canvas.draw()
        img_arr = list(f.axes[0].get_images())
        ax.set_title(f'Token: "{tok}"', fontsize=8)
        ax.axis('off')
    plt.suptitle(f'Similarity maps for "{query_vis}"', fontsize=11)
    plt.tight_layout()
    plt.show()
    print('Interpretability visualization complete')

except ImportError:
    print('Install colpali-engine[interpretability] to see similarity maps')
    print('pip install "colpali-engine[interpretability]"')

In [ ]:
# ── 8. Run the evaluation suite ───────────────────────────
from src.evaluation.evaluator import RAGEvaluator

# only run 2 queries in the notebook to save time
from src.evaluation.evaluator import BENCHMARK_QUERIES
mini_bench = BENCHMARK_QUERIES[:2]

evaluator = RAGEvaluator(index_dir='../index/', top_k=3)
report = evaluator.run(queries=mini_bench, output_path='../eval_results.json')

print(f'\nAvg faithfulness : {report.avg_faithfulness:.1%}')
print(f'Avg retrieval    : {report.avg_retrieval_score:.3f}')